# Go1 IMU sensitivity — Colab runner

1. **Train** a Go1 joystick policy.
2. **OFAT sweep** — inject idealised IMU errors one at a time, find where
   performance knees, compare each knee against a real BMI270.

The sweep is the decision point. If no channel's knee is reachable by a real
device, the project's premise is false for this task and we learn it here.

**Set Runtime → Change runtime type → GPU before running anything.**

## 1. GPU check

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

## 2. Install

Do NOT install jax here. Colab ships a working CUDA-enabled JAX; upgrading it
leaves the preinstalled cuda13 plugin at the old version and both plugins then
fight to register.

In [ ]:
# JAX MUST BE PINNED TO 0.10.x. Colab ships 0.11.x, and brax 0.14.2 -- the
# newest release -- calls jax.device_put_replicated, removed in 0.11. There is
# no brax that supports 0.11 yet, so the ecosystem pins the version, not Colab.
#
# Uninstall the CUDA plugins first: a partial upgrade leaves a stale plugin
# behind and both then race to register (cuda13 API mismatch, cuda12
# ALREADY_EXISTS). Installing jax[cuda12]==0.10.2 brings a matching plugin.
#
# flax>=0.12.8 is also required: older flax calls jax.core.get_opaque_trace_state.
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda13-plugin
!pip install -q "jax[cuda12]==0.10.2"
!pip install -q mujoco mujoco-mjx brax playground "flax>=0.12.8"

print("\n*** Runtime > Restart session now, then continue from the next cell. ***")

In [ ]:
import jax, jaxlib, flax, brax
from packaging.version import Version

for n, v in (("jax", jax.__version__), ("jaxlib", jaxlib.__version__),
             ("flax", flax.__version__), ("brax", brax.__version__)):
    print(f"{n:8s} {v}")
devs = jax.devices()
print("devices:", devs)

assert any(d.platform == "gpu" for d in devs), (
    "No GPU visible to JAX. Runtime > Change runtime type > GPU, then "
    "Runtime > Restart session, then re-run from the top."
)
assert Version(jax.__version__) < Version("0.11"), (
    f"jax {jax.__version__} is too new for brax {brax.__version__}, which calls "
    "jax.device_put_replicated (removed in 0.11). Training dies at "
    "device_put_replicated. Re-run the install cell and restart the session."
)
assert Version(flax.__version__) >= Version("0.12.8"), (
    f"flax {flax.__version__} is too old: it calls jax.core.get_opaque_trace_state, "
    "removed in JAX 0.11. Training dies at network init."
)
print("\nOK -- GPU visible, jax/flax/brax mutually compatible")

## 3. Clone

The repo is public, so no token and no secrets setup.

In [ ]:
!git clone -q https://github.com/stosifian/imu-diff-sim.git /content/imu-diff-sim 2>/dev/null || true
%cd /content/imu-diff-sim
!git pull
!pip install -q -e .
!echo "--- now at ---" && git log --oneline -1

## 4. Where checkpoints go

`/content` is wiped when the VM disconnects (~90 min idle, 12 h max). For a
short run that is fine; for a multi-hour run, set `USE_DRIVE = True`.

In [ ]:
USE_DRIVE = False   # True for long runs you cannot babysit

import pathlib

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = pathlib.Path('/content/drive/MyDrive/imu-diff-sim-runs')
else:
    OUT = pathlib.Path('/content/runs')
    print("NOT using Drive: these files vanish when the VM disconnects.")
    print("Download anything you want to keep before you walk away.")

OUT.mkdir(parents=True, exist_ok=True)
print("checkpoints ->", OUT)

## 5. Sanity check

64 tests. Expect ~12 min on Colab -- slower than local despite the GPU,
because the 2M-sample Allan traces are CPU-bound NumPy and Colab's CPU is
weaker than an M1.

In [ ]:
!python -m pytest -q --no-header -x

## 6. Train

`clean` (`--noise-level 0.0`) is the one the OFAT needs: injected errors must
be measured against a clean floor, not on top of the stock ±0.2 rad/s gyro DR
that is already 146x a real BMI270.

Run the throughput cell first and watch steps/sec and the reward. Reward flat
after a few million steps means the install is wrong, not the task.

In [ ]:
!python scripts/train_go1.py --timesteps 2000000 --noise-level 0.0 --out {OUT}/go1_clean_smoke.pkl

In [ ]:
!python scripts/train_go1.py --noise-level 0.0 --out {OUT}/go1_clean.pkl

In [ ]:
# Only needed for the stock-vs-clean comparison; skip on a first pass.
!python scripts/train_go1.py --noise-level 1.0 --out {OUT}/go1_stock.pkl

## 7. OFAT sweep

Watch `gyro_white` against `gyro_bias` at equal magnitude. White noise the
policy can average away over an episode (as 1/sqrt(N)); a persistent bias it
cannot. If bias knees at far lower magnitude, the structural argument is
confirmed empirically rather than asserted.

In [ ]:
!python scripts/ofat_sensitivity.py --policy {OUT}/go1_clean.pkl \
    --episodes 64 --horizon 500 --points 9 --out {OUT}/ofat_clean.npz

## 8. Read the result

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from imu_diff_sim.sim.go1_obs import (
    BMI270_GYRO_WHITE_STD_50HZ, PLAYGROUND_GYRO_DR_STD,
    BMI270_GYRO_TURN_ON_BIAS_BOUND, PLAYGROUND_GYRO_DR_EPISODE_MEAN_500)

d = np.load(OUT / 'ofat_clean.npz')
factors = sorted({k.split('__')[0] for k in d.files})
fig, axes = plt.subplots(1, len(factors), figsize=(4*len(factors), 4),
                         constrained_layout=True)
for ax, f in zip(np.atleast_1d(axes), factors):
    m, r = d[f'{f}__magnitudes'], d[f'{f}__return']
    ax.plot(m, r / max(r.max(), 1e-9), 'o-')
    if f == 'gyro_white':
        ax.axvline(BMI270_GYRO_WHITE_STD_50HZ, color='green', ls='--', label='BMI270')
        ax.axvline(PLAYGROUND_GYRO_DR_STD, color='crimson', ls=':', label='DR')
    if f == 'gyro_bias':
        ax.axvline(BMI270_GYRO_TURN_ON_BIAS_BOUND, color='green', ls='--',
                   label='BMI270 turn-on bias')
        ax.axvline(PLAYGROUND_GYRO_DR_EPISODE_MEAN_500, color='crimson', ls=':',
                   label='DR persistent part')
    if f.startswith('gyro'):
        ax.legend(fontsize=7)
    if f != 'latency':
        ax.set_xscale('log')
    ax.set_title(f); ax.set_xlabel('magnitude')
    ax.set_ylabel('normalised return'); ax.grid(alpha=.3)
plt.show()

In [ ]:
# Keep anything you care about before the VM disconnects.
from google.colab import files
files.download(str(OUT / 'ofat_clean.npz'))